In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

### Load Dataset

In [3]:
calendar = pd.read_csv("../data/calendar.csv")

sales = pd.read_csv("../data/sales_train_validation.csv")

In [4]:
calendar.shape, sales.shape

((1969, 14), (30490, 1919))

### Create Daily Sales Dataset

In [5]:
daily_columns = [
    col for col in sales.columns
    if col.startswith("d_")
]

daily_sales = (
    sales[daily_columns]
    .sum(axis=0)
    .reset_index()
)

daily_sales.columns = ["d", "sales"]

daily_sales.head()

,d,sales
0,d_1,32631
1,d_2,31749
2,d_3,23783
3,d_4,25412
4,d_5,19146


### : Merge Calendar

In [7]:
daily_sales = daily_sales.merge(
    calendar[['d', 'date']],
    on='d',
    how='left'
)

### Convert Date

In [9]:
daily_sales['date'] = pd.to_datetime(
    daily_sales['date']
)

daily_sales.head()

,d,sales,date
0,d_1,32631,2011-01-29
1,d_2,31749,2011-01-30
2,d_3,23783,2011-01-31
3,d_4,25412,2011-02-01
4,d_5,19146,2011-02-02


### Create Modeling Dataset

In [11]:
df = daily_sales.copy()

df.head()

,d,sales,date
0,d_1,32631,2011-01-29
1,d_2,31749,2011-01-30
2,d_3,23783,2011-01-31
3,d_4,25412,2011-02-01
4,d_5,19146,2011-02-02


### Calendar Features

In [12]:
df['year'] = df['date'].dt.year

df['month'] = df['date'].dt.month

df['quarter'] = df['date'].dt.quarter

df['day'] = df['date'].dt.day

df['weekday'] = df['date'].dt.weekday

df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)

In [13]:
df.head()

,d,sales,date,year,month,quarter,day,weekday,weekofyear
0,d_1,32631,2011-01-29,2011,1,1,29,5,4
1,d_2,31749,2011-01-30,2011,1,1,30,6,4
2,d_3,23783,2011-01-31,2011,1,1,31,0,5
3,d_4,25412,2011-02-01,2011,2,1,1,1,5
4,d_5,19146,2011-02-02,2011,2,1,2,2,5


### Weekend Flag

In [15]:
df['is_weekend'] = np.where(
    df['weekday'] >= 5,
    1,
    0
)

df[['date','weekday','is_weekend']].head()

,date,weekday,is_weekend
0,2011-01-29,5,1
1,2011-01-30,6,1
2,2011-01-31,0,0
3,2011-02-01,1,0
4,2011-02-02,2,0


### Event Features

In [17]:
calendar['is_event'] = np.where(
    calendar['event_type_1'].notna(),
    1,
    0
)

In [18]:
calendar[['event_type_1', 'is_event']].head()

,event_type_1,is_event
0,NaN,0
1,NaN,0
2,NaN,0
3,NaN,0
4,NaN,0


In [20]:
df['date'].dtype

dtype('<M8[ns]')

In [21]:
calendar['date'].dtype

dtype('O')

In [22]:
calendar['date'] = pd.to_datetime(calendar['date'])

In [23]:
event_features = calendar[
    ['date', 'is_event', 'event_type_1']
].copy()

df = df.merge(
    event_features,
    on='date',
    how='left'
)

In [24]:
df['event_type_1'] = df['event_type_1'].fillna('No_Event')

In [25]:
df.head()

,d,sales,date,year,month,quarter,day,weekday,weekofyear,is_weekend,is_event,event_type_1
0,d_1,32631,2011-01-29,2011,1,1,29,5,4,1,0,No_Event
1,d_2,31749,2011-01-30,2011,1,1,30,6,4,1,0,No_Event
2,d_3,23783,2011-01-31,2011,1,1,31,0,5,0,0,No_Event
3,d_4,25412,2011-02-01,2011,2,1,1,1,5,0,0,No_Event
4,d_5,19146,2011-02-02,2011,2,1,2,2,5,0,0,No_Event


### Lag Features

In [26]:
df['sales_lag_7'] = df['sales'].shift(7)

In [27]:
df['sales_lag_14'] = df['sales'].shift(14)

In [28]:
df['sales_lag_28'] = df['sales'].shift(28)

In [29]:
df[
    ['sales',
     'sales_lag_7',
     'sales_lag_14',
     'sales_lag_28']
].head(35)

,sales,sales_lag_7,sales_lag_14,sales_lag_28
0,32631,NaN,NaN,NaN
1,31749,NaN,NaN,NaN
2,23783,NaN,NaN,NaN
3,25412,NaN,NaN,NaN
4,19146,NaN,NaN,NaN
5,29211,NaN,NaN,NaN
6,28010,NaN,NaN,NaN
7,37932,32631.0,NaN,NaN
8,32736,31749.0,NaN,NaN
9,25572,23783.0,NaN,NaN


### Rolling Features

In [30]:
df['rolling_mean_7'] = (
    df['sales']
    .shift(1)
    .rolling(7)
    .mean()
)

In [31]:
df['rolling_mean_14'] = (
    df['sales']
    .shift(1)
    .rolling(14)
    .mean()
)

In [32]:
df['rolling_mean_28'] = (
    df['sales']
    .shift(1)
    .rolling(28)
    .mean()
)

### Rolling Std Features

In [33]:
df['rolling_std_7'] = (
    df['sales']
    .shift(1)
    .rolling(7)
    .std()
)

df['rolling_std_28'] = (
    df['sales']
    .shift(1)
    .rolling(28)
    .std()
)

In [34]:
### Check Missing Values

In [35]:
df.isnull().sum()

d                   0
sales               0
date                0
year                0
month               0
quarter             0
day                 0
weekday             0
weekofyear          0
is_weekend          0
is_event            0
event_type_1        0
sales_lag_7         7
sales_lag_14       14
sales_lag_28       28
rolling_mean_7      7
rolling_mean_14    14
rolling_mean_28    28
rolling_std_7       7
rolling_std_28     28
dtype: int64

In [36]:
df = df.dropna()

df.shape

(1885, 20)

### Save Feature Dataset

In [38]:
df.to_csv(
    "../data/processed/feature_engineered_data.csv",
    index=False
)

In [39]:
df.shape

(1885, 20)

In [40]:
df.head()

,d,sales,date,year,month,quarter,day,weekday,weekofyear,is_weekend,is_event,event_type_1,sales_lag_7,sales_lag_14,sales_lag_28,rolling_mean_7,rolling_mean_14,rolling_mean_28,rolling_std_7,rolling_std_28
28,d_29,29908,2011-02-26,2011,2,1,26,5,8,1,0,No_Event,31689.0,34833.0,32631.0,24143.142857,25112.214286,26238.678571,4576.628250,5320.795395
29,d_30,28707,2011-02-27,2011,2,1,27,6,8,1,0,No_Event,29283.0,36380.0,31749.0,23888.714286,24760.428571,26141.428571,4113.263859,5223.630967
30,d_31,21240,2011-02-28,2011,2,1,28,0,9,0,0,No_Event,23966.0,21804.0,23783.0,23806.428571,24212.357143,26032.785714,3991.319742,5133.540619
31,d_32,22872,2011-03-01,2011,3,1,1,1,9,0,0,No_Event,20501.0,24070.0,25412.0,23417.000000,24172.071429,25941.964286,4104.536312,5196.921314
32,d_33,22046,2011-03-02,2011,3,1,2,2,9,0,0,No_Event,20757.0,21443.0,19146.0,23755.714286,24086.500000,25851.250000,3917.358537,5228.586542


In [41]:
df.columns

Index(['d', 'sales', 'date', 'year', 'month', 'quarter', 'day', 'weekday',
       'weekofyear', 'is_weekend', 'is_event', 'event_type_1', 'sales_lag_7',
       'sales_lag_14', 'sales_lag_28', 'rolling_mean_7', 'rolling_mean_14',
       'rolling_mean_28', 'rolling_std_7', 'rolling_std_28'],
      dtype='object')

## Feature Engineering Summary

Created calendar-based, event-based, lag-based and rolling statistical features to capture seasonality, trend, and historical demand patterns for forecasting models.

Features include:
- Date Features
- Event Features
- Lag Features
- Rolling Mean Features
- Rolling Standard Deviation Features